# GMAO-RAG API - Tests Manuels Complets

Ce notebook teste **tous les endpoints** de l'API REST FastAPI (port 8000).
Chaque cellule teste un ou plusieurs cas (succès, erreur, validation, auth).

## Prérequis
1. Lancer le serveur : `uvicorn app.api.main:app --host 0.0.0.0 --port 8000 --reload`
2. Avoir MySQL et Qdrant actifs
3. Exécuter les cellules dans l'ordre

## Endpoints couverts

| Méthode | Endpoint | Auth | Description |
|---------|----------|------|-------------|
| GET | `/api/v1/health` | Non | Health check |
| GET | `/api/v1/strategies` | Oui | Stratégies disponibles |
| GET | `/api/v1/stats` | Oui | Statistiques pipeline |
| GET | `/api/v1/documents` | Oui | Liste documents |
| GET | `/api/v1/documents/{id}` | Oui | Détail document |
| DELETE | `/api/v1/documents/{id}` | Oui | Supprimer document |
| POST | `/api/v1/rag/retrieve` | Oui | Retrieval seul |
| POST | `/api/v1/rag/rerank` | Oui | Reranking seul |
| POST | `/api/v1/rag/search` | Oui | Pipeline complet |
| POST | `/api/v1/ingest/file` | Oui | Upload fichier |
| POST | `/api/v1/ingest/database` | Oui | Ingestion MySQL |
| POST | `/api/v1/ingest/files` | Oui | Batch ingestion |

---
## 0. Configuration

> **Attention** : adapter `API_KEY` selon votre fichier `.env` (`RAG_API_KEY`).

In [4]:
import requests
import json
import time
from pathlib import Path

# ============================================================
# Configuration
# ============================================================
BASE_URL = "http://localhost:8000"
API_KEY = "Zzdv0632"  # Clé RAG_API_KEY du .env

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

def show(label, resp):
    """Affiche le status code et le body JSON."""
    print(f"\n{"="*60}")
    print(f"  {label}")
    print(f"  Status: {resp.status_code}")
    pt = resp.headers.get("X-Process-Time", "N/A")
    print(f"  X-Process-Time: {pt}s")
    print(f"  Body:")
    try:
        print(json.dumps(resp.json(), indent=2, ensure_ascii=False)[:3000])
    except Exception:
        print(resp.text[:3000])
    print(f"{"="*60}")

print(f"Configuration OK - BASE_URL: {BASE_URL}")

Configuration OK - BASE_URL: http://localhost:8000


---
## 1. Health Check (GET /api/v1/health)

**Pas d'authentification requise.** Vérifie la connectivité Qdrant + MySQL.
Retourne `status`: `healthy`, `degraded` ou `unhealthy`.

In [52]:
# --- Cas 1.1 : Health check normal ---
# Attendu: 200 + {status, qdrant, mysql, version}
resp = requests.get(f"{BASE_URL}/api/v1/health")
show("GET /api/v1/health - Cas normal", resp)

# Vérifier les champs obligatoires
data = resp.json()
assert "status" in data
assert "qdrant" in data
assert "mysql" in data
assert "version" in data
print(f"\nOK - status={data['status']}, qdrant={data['qdrant']}, mysql={data['mysql']}")


  GET /api/v1/health - Cas normal
  Status: 200
  X-Process-Time: 0.2473s
  Body:
{
  "status": "healthy",
  "qdrant": "ok",
  "mysql": "ok",
  "version": "0.1.0"
}

OK - status=healthy, qdrant=ok, mysql=ok


---
## 2. Stratégies (GET /api/v1/strategies)

**Auth requise.** Retourne les noms des stratégies enregistrées par couche du pipeline.

In [ ]:
# --- Cas 2.1 : Auth valide ---
# Attendu: 200 + {retrieval[], reranker[], llm[], embedding[]}
resp = requests.get(f"{BASE_URL}/api/v1/strategies", headers=HEADERS)
show("GET /api/v1/strategies - Auth valide", resp)
data = resp.json()
for key in ["retrieval", "reranker", "llm", "embedding"]:
    assert key in data and isinstance(data[key], list)
print(f"\nOK - retrieval={data['retrieval']}, reranker={data['reranker']}, llm={data['llm']}")


  GET /api/v1/strategies - Auth valide
  Status: 200
  X-Process-Time: 0.0092s
  Body:
{
  "retrieval": [
    "hybrid",
    "qdrant"
  ],
  "reranker": [
    "cross-encoder"
  ],
  "llm": [
    "gemini",
    "openai"
  ],
  "embedding": [
    "sentence-transformer"
  ]
}

OK - retrieval=['hybrid', 'qdrant'], reranker=['cross-encoder'], llm=['gemini', 'openai']


In [ ]:
# --- Cas 2.2 : Sans auth ---
# Attendu: 422 (header requis)

resp = requests.get(f"{BASE_URL}/api/v1/strategies")
show("GET /api/v1/strategies - Sans auth", resp)

# --- Cas 2.3 : Mauvais token ---
# Attendu: 401
resp = requests.get(f"{BASE_URL}/api/v1/strategies", headers={"Authorization": "Bearer wrong-token"})
show("GET /api/v1/strategies - Mauvais token", resp)
assert resp.status_code == 401

# --- Cas 2.4 : Header mal formé (pas de Bearer) ---
# Attendu: 401
resp = requests.get(f"{BASE_URL}/api/v1/strategies", headers={"Authorization": "Token abc"})
show("GET /api/v1/strategies - Header mal formé", resp)
assert resp.status_code == 401


  GET /api/v1/strategies - Sans auth
  Status: 422
  X-Process-Time: 0.0055s
  Body:
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "header",
        "Authorization"
      ],
      "msg": "Field required",
      "input": null
    }
  ]
}

  GET /api/v1/strategies - Mauvais token
  Status: 401
  X-Process-Time: 0.0014s
  Body:
{
  "detail": "Invalid API key."
}

  GET /api/v1/strategies - Header mal formé
  Status: 401
  X-Process-Time: 0.0014s
  Body:
{
  "detail": "Missing or malformed Authorization header. Expected 'Bearer <token>'."
}


---
## 3. Statistiques (GET /api/v1/stats)

**Auth requise.** Nombre de documents, chunks, et points Qdrant.

In [9]:
# --- Cas 3.1 : Stats normales ---
# Attendu: 200 + {documents_count, chunks_count, qdrant_points}
resp = requests.get(f"{BASE_URL}/api/v1/stats", headers=HEADERS)
show("GET /api/v1/stats - Cas normal", resp)
data = resp.json()
assert data["documents_count"] >= 0 and data["chunks_count"] >= 0

# --- Cas 3.2 : Sans auth ---
# Attendu: 422
resp = requests.get(f"{BASE_URL}/api/v1/stats")
show("GET /api/v1/stats - Sans auth", resp)


  GET /api/v1/stats - Cas normal
  Status: 200
  X-Process-Time: 0.3323s
  Body:
{
  "documents_count": 1,
  "chunks_count": 52,
  "qdrant_points": 83
}

  GET /api/v1/stats - Sans auth
  Status: 422
  X-Process-Time: 0.0024s
  Body:
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "header",
        "Authorization"
      ],
      "msg": "Field required",
      "input": null
    }
  ]
}


---
## 4. Documents (GET / DELETE `/api/v1/documents`)

**Auth requise.** Gestion des documents indexés dans MySQL.

In [13]:
# --- Cas 4.1 : Liste des documents ---
# Attendu: 200 + {documents[], total}
resp = requests.get(f"{BASE_URL}/api/v1/documents", headers=HEADERS)
show("GET /api/v1/documents - Liste", resp)
data = resp.json()
assert "documents" in data and "total" in data
print(f"\nOK - {data['total']} document(s)")


  GET /api/v1/documents - Liste
  Status: 200
  X-Process-Time: 0.4146s
  Body:
{
  "documents": [
    {
      "id": 1,
      "name": "sample-5pages.docx",
      "source_type": "DOCX",
      "chunks_count": 19,
      "indexed": false
    }
  ],
  "total": 1
}

OK - 1 document(s)


In [14]:
# --- Cas 4.2 : Détail document existant ---
doc_list = resp.json().get("documents", [])
if doc_list:
    doc_id = doc_list[0]["id"]
    resp2 = requests.get(f"{BASE_URL}/api/v1/documents/{doc_id}", headers=HEADERS)
    show(f"GET /api/v1/documents/{doc_id} - Détail", resp2)
    detail = resp2.json()
    print(f"  -> {detail['document']['chunks_count']} chunk(s)")
else:
    print("Aucun document dans la base - test détails ignoré")

# --- Cas 4.3 : Document inexistant ---
# Attendu: 404
resp = requests.get(f"{BASE_URL}/api/v1/documents/999999", headers=HEADERS)
show("GET /api/v1/documents/999999 - Inexistant", resp)
assert resp.status_code == 404

# --- Cas 4.4 : ID non entier ---
# Attendu: 422
resp = requests.get(f"{BASE_URL}/api/v1/documents/abc", headers=HEADERS)
show("GET /api/v1/documents/abc - ID invalide", resp)
assert resp.status_code == 422


  GET /api/v1/documents/1 - Détail
  Status: 200
  X-Process-Time: 0.4158s
  Body:
{
  "document": {
    "id": 1,
    "name": "sample-5pages.docx",
    "source_type": "DOCX",
    "chunks_count": 19,
    "indexed": false
  },
  "chunks": [
    {
      "chunk_id": "69",
      "content": "Introduction to Digital Documents\nDigital documents have become the backbone of modern communication. From business reports to academic papers, electronic files allow people to share information quickly and reliably across the globe.",
      "score": 0.0,
      "rank": 0,
      "source_name": "sample-5pages.docx",
      "source_type": "Document",
      "id_document": 1,
      "id_panne": null,
      "id_equipement": null,
      "retrieval_strategy": ""
    },
    {
      "chunk_id": "86",
      "content": "--- Row 1 ---\nid_document: 1\ntitre: Sample document – 5 pages\nnom_fichier: sample-5pages.docx\ntype_fichier: DOCX\nchemin_fichier: /home/abdellah-daif/GMAO-RAG/tests/data/docx/sample-5pages.docx\n

In [15]:
resp = requests.get(f"{BASE_URL}/api/v1/documents/999999", headers=HEADERS)
show("GET /api/v1/documents/999999 - Inexistant", resp)
assert resp.status_code == 404


  GET /api/v1/documents/999999 - Inexistant
  Status: 404
  X-Process-Time: 0.1271s
  Body:
{
  "detail": "Document 999999 not found."
}


In [16]:
# --- Cas 4.4 : ID non entier ---
# Attendu: 422
resp = requests.get(f"{BASE_URL}/api/v1/documents/abc", headers=HEADERS)
show("GET /api/v1/documents/abc - ID invalide", resp)
assert resp.status_code == 422


  GET /api/v1/documents/abc - ID invalide
  Status: 422
  X-Process-Time: 0.0050s
  Body:
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": [
        "path",
        "document_id"
      ],
      "msg": "Input should be a valid integer, unable to parse string as an integer",
      "input": "abc"
    }
  ]
}


In [18]:
# --- Cas 4.5 : DELETE document inexistant ---
# Attendu: 404
resp = requests.delete(f"{BASE_URL}/api/v1/documents/999999", headers=HEADERS)
show("DELETE /api/v1/documents/999999 - Inexistant", resp)
assert resp.status_code == 404

# --- Cas 4.6 : DELETE document existant (DESTRUCTIF !) ---
# ATTENTION : Décommenter uniquement si vous voulez supprimer un document !
resp = requests.delete(f"{BASE_URL}/api/v1/documents/{doc_id}", headers=HEADERS)
show(f"DELETE /api/v1/documents/{doc_id} - Suppression", resp)


  DELETE /api/v1/documents/999999 - Inexistant
  Status: 404
  X-Process-Time: 0.1031s
  Body:
{
  "detail": "Document 999999 not found."
}

  DELETE /api/v1/documents/1 - Suppression
  Status: 200
  X-Process-Time: 0.1519s
  Body:
{
  "status": "ok",
  "deleted_chunks": 19
}


---
## 5. Retrieval (POST /api/v1/rag/retrieve)

**Auth requise.** Recherche de chunks pertinents sans reranking ni LLM.

In [19]:
# --- Cas 5.1 : Retrieval simple ---
# Attendu: 200 + {query, results[], total_candidates, strategy_name}
payload = {"query": "pompe vibration", "top_k": 5}
resp = requests.post(f"{BASE_URL}/api/v1/rag/retrieve", headers=HEADERS, json=payload)
show("POST /api/v1/rag/retrieve - Cas simple", resp)
data = resp.json()
assert "results" in data and "strategy_name" in data
print(f"\n  -> {len(data['results'])} résultats, stratégie: {data['strategy_name']}")


  POST /api/v1/rag/retrieve - Cas simple
  Status: 200
  X-Process-Time: 0.1576s
  Body:
{
  "query": "pompe vibration",
  "results": [
    {
      "chunk_id": "112",
      "content": "--- Row 1 ---\nid_panne: 7\ntitre: Vibration anormale sur pompe centrifuge\ndescription: Niveau sonore excessif et secousses importantes mesurées sur le corps de pompe.\ngravite: Moyenne\ndate_detection: 2026-06-07 15:50:00",
      "score": 0.8797649,
      "rank": 1,
      "source_name": "panne:7",
      "source_type": "panne",
      "id_document": null,
      "id_panne": 7,
      "id_equipement": 224,
      "retrieval_strategy": "qdrant"
    },
    {
      "chunk_id": "114",
      "content": "réalignement laser de l'arbre.\nsymptomes: Bruit de cognement métallique, oscillations sur les capteurs de vibration.\nstatut_indexation: En_attente\nid_equipement: 224\nid_ot: 5053",
      "score": 0.8203136,
      "rank": 2,
      "source_name": "panne:7",
      "source_type": "panne",
      "id_document": null

In [21]:
# --- Cas 5.2 : Avec filtres source_type + min_score ---
payload = {
    "query": "panne moteur",
    "filters": {"source_type": "panne", "min_score": 0.5},
    "top_k": 10
}
resp = requests.post(f"{BASE_URL}/api/v1/rag/retrieve", headers=HEADERS, json=payload)
show("POST /api/v1/rag/retrieve - Filtres source_type + min_score", resp)

# --- Cas 5.3 : Filtre par id_equipement ---
payload = {"query": "maintenance", "filters": {"id_equipement": 1}, "top_k": 5}
resp = requests.post(f"{BASE_URL}/api/v1/rag/retrieve", headers=HEADERS, json=payload)
show("POST /api/v1/rag/retrieve - Filtre id_equipement", resp)


  POST /api/v1/rag/retrieve - Filtres source_type + min_score
  Status: 200
  X-Process-Time: 0.1791s
  Body:
{
  "query": "panne moteur",
  "results": [
    {
      "chunk_id": "88",
      "content": "--- Row 1 ---\nid_panne: 1\ntitre: Surchauffe moteur principal\ndescription: Le moteur principal a déclenché une alarme de température élevée lors d'un cycle de production en continu.\ngravite: Critique\ndate_detection: 2026-06-01 08:30:00",
      "score": 0.86970806,
      "rank": 1,
      "source_name": "panne:1",
      "source_type": "panne",
      "id_document": null,
      "id_panne": 1,
      "id_equipement": 102,
      "retrieval_strategy": "qdrant"
    },
    {
      "chunk_id": "94",
      "content": "--- Row 1 ---\nid_panne: 1\ntitre: Surchauffe moteur principal\ndescription: Le moteur principal a déclenché une alarme de température élevée lors d'un cycle de production en continu.\ngravite: Critique\ndate_detection: 2026-06-01 08:30:00",
      "score": 0.86970806,
      "rank"

In [23]:
# --- Cas 5.4 : Query vide ---
# Attendu: 422 (min_length=1)
resp = requests.post(f"{BASE_URL}/api/v1/rag/retrieve", headers=HEADERS, json={"query": ""})
show("POST /api/v1/rag/retrieve - Query vide", resp)
assert resp.status_code == 422

# -


  POST /api/v1/rag/retrieve - Query vide
  Status: 422
  X-Process-Time: 0.0051s
  Body:
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "body",
        "query"
      ],
      "msg": "String should have at least 1 character",
      "input": "",
      "ctx": {
        "min_length": 1
      }
    }
  ]
}


In [24]:
# -- Cas 5.5 : top_k = 0 ---
# Attendu: 422 (ge=1)
resp = requests.post(f"{BASE_URL}/api/v1/rag/retrieve", headers=HEADERS, json={"query": "test", "top_k": 0})
show("POST /api/v1/rag/retrieve - top_k=0", resp)
assert resp.status_code == 422




  POST /api/v1/rag/retrieve - top_k=0
  Status: 422
  X-Process-Time: 0.0066s
  Body:
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "body",
        "top_k"
      ],
      "msg": "Input should be greater than or equal to 1",
      "input": 0,
      "ctx": {
        "ge": 1
      }
    }
  ]
}


In [25]:
# --- Cas 5.6 : top_k = 100 ---
# Attendu: 422 (le=50)
resp = requests.post(f"{BASE_URL}/api/v1/rag/retrieve", headers=HEADERS, json={"query": "test", "top_k": 100})
show("POST /api/v1/rag/retrieve - top_k=100", resp)
assert resp.status_code == 422




  POST /api/v1/rag/retrieve - top_k=100
  Status: 422
  X-Process-Time: 0.0070s
  Body:
{
  "detail": [
    {
      "type": "less_than_equal",
      "loc": [
        "body",
        "top_k"
      ],
      "msg": "Input should be less than or equal to 50",
      "input": 100,
      "ctx": {
        "le": 50
      }
    }
  ]
}


In [28]:
# --- Cas 5.7 : Sans auth ---
# resp = requests.post(f"{BASE_URL}/api/v1/rag/retrieve", json={"query": "test"})
# show("POST /api/v1/rag/retrieve - Sans auth", resp)

# --- Cas 5.8 : Body manquant ---
resp = requests.post(f"{BASE_URL}/api/v1/rag/retrieve", headers=HEADERS)
show("POST /api/v1/rag/retrieve - Body manquant", resp)


  POST /api/v1/rag/retrieve - Body manquant
  Status: 422
  X-Process-Time: 0.0047s
  Body:
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body"
      ],
      "msg": "Field required",
      "input": null
    }
  ]
}


---
## 6. Rerank (POST /api/v1/rag/rerank)

**Auth requise.** Re-classement des chunks via cross-encoder.
On récupère d'abord des chunks via `/retrieve`, puis on les passe à `/rerank`.

In [29]:
# --- Cas 6.1 : Rerank normal ---
# Récupérer des chunks d'abord
retrieve_resp = requests.post(
    f"{BASE_URL}/api/v1/rag/retrieve",
    headers=HEADERS,
    json={"query": "pompe vibration", "top_k": 5}
)
chunks = retrieve_resp.json().get("results", [])

if chunks:
    payload = {"query": "pompe vibration", "candidates": chunks, "top_k": 3}
    resp = requests.post(f"{BASE_URL}/api/v1/rag/rerank", headers=HEADERS, json=payload)
    show("POST /api/v1/rag/rerank - Cas normal", resp)
    data = resp.json()
    for r in data["results"]:
        print(f"  rank={r['rank']} rerank_score={r['rerank_score']:.4f} | {r['content'][:60]}...")
else:
    print("Pas de chunks récupérés - test rerank ignoré")


  POST /api/v1/rag/rerank - Cas normal
  Status: 200
  X-Process-Time: 18.0855s
  Body:
{
  "query": "pompe vibration",
  "results": [
    {
      "chunk_id": "112",
      "content": "--- Row 1 ---\nid_panne: 7\ntitre: Vibration anormale sur pompe centrifuge\ndescription: Niveau sonore excessif et secousses importantes mesurées sur le corps de pompe.\ngravite: Moyenne\ndate_detection: 2026-06-07 15:50:00",
      "source_name": "panne:7",
      "source_type": "panne",
      "retrieval_score": 0.8797649,
      "rerank_score": 0.9529886245727539,
      "rank": 1,
      "id_document": null,
      "id_panne": 7,
      "id_equipement": 224,
      "retrieval_strategy": "qdrant",
      "reranker_strategy": "cross-encoder"
    },
    {
      "chunk_id": "114",
      "content": "réalignement laser de l'arbre.\nsymptomes: Bruit de cognement métallique, oscillations sur les capteurs de vibration.\nstatut_indexation: En_attente\nid_equipement: 224\nid_ot: 5053",
      "source_name": "panne:7",
   

In [31]:
# --- Cas 6.2 : Candidats vides ---
# Attendu: 422 (min_length=1)
resp = requests.post(f"{BASE_URL}/api/v1/rag/rerank", headers=HEADERS,
    json={"query": "test", "candidates": []})
show("POST /api/v1/rag/rerank - Candidats vides", resp)
assert resp.status_code == 422






  POST /api/v1/rag/rerank - Candidats vides
  Status: 422
  X-Process-Time: 0.0180s
  Body:
{
  "detail": [
    {
      "type": "too_short",
      "loc": [
        "body",
        "candidates"
      ],
      "msg": "List should have at least 1 item after validation, not 0",
      "input": [],
      "ctx": {
        "field_type": "List",
        "min_length": 1,
        "actual_length": 0
      }
    }
  ]
}


In [32]:
# --- Cas 6.3 : Query vide ---
# Attendu: 422
resp = requests.post(f"{BASE_URL}/api/v1/rag/rerank", headers=HEADERS, json={
    "query": "",
    "candidates": [{"chunk_id": "c1", "content": "x", "score": 0.5, "rank": 1,
                    "source_name": "test.txt", "source_type": "document"}]
})
show("POST /api/v1/rag/rerank - Query vide", resp)
assert resp.status_code == 422


  POST /api/v1/rag/rerank - Query vide
  Status: 422
  X-Process-Time: 0.0041s
  Body:
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "body",
        "query"
      ],
      "msg": "String should have at least 1 character",
      "input": "",
      "ctx": {
        "min_length": 1
      }
    }
  ]
}


In [33]:
# --- Cas 6.4 : Candidate avec champs manquants ---
# Attendu: 422
resp = requests.post(f"{BASE_URL}/api/v1/rag/rerank", headers=HEADERS,
    json={"query": "test", "candidates": [{"chunk_id": "only-id"}]})
show("POST /api/v1/rag/rerank - Champs manquants", resp)
assert resp.status_code == 422


  POST /api/v1/rag/rerank - Champs manquants
  Status: 422
  X-Process-Time: 0.0062s
  Body:
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "candidates",
        0,
        "content"
      ],
      "msg": "Field required",
      "input": {
        "chunk_id": "only-id"
      }
    },
    {
      "type": "missing",
      "loc": [
        "body",
        "candidates",
        0,
        "score"
      ],
      "msg": "Field required",
      "input": {
        "chunk_id": "only-id"
      }
    },
    {
      "type": "missing",
      "loc": [
        "body",
        "candidates",
        0,
        "rank"
      ],
      "msg": "Field required",
      "input": {
        "chunk_id": "only-id"
      }
    },
    {
      "type": "missing",
      "loc": [
        "body",
        "candidates",
        0,
        "source_name"
      ],
      "msg": "Field required",
      "input": {
        "chunk_id": "only-id"
      }
    },
    {
      "type": "missing",


---
## 7. Search - Pipeline complet (POST /api/v1/rag/search)

**Auth requise.** Retrieve -> Rerank -> Generate (LLM).
Possibilité de désactiver `rerank` ou `generate` pour des variantes plus rapides.

In [38]:
# --- Cas 7.1 : Pipeline complet ---
# Attendu: 200 + {answer, citations[], results[], strategy_info, duration_ms}
payload = {
    "query": "Quelles sont les causes fréquentes de panne sur les pompes ?",
    "top_k": 5,
    "rerank": True,
    "generate": True
}
resp = requests.post(f"{BASE_URL}/api/v1/rag/search", headers=HEADERS, json=payload)
show("POST /api/v1/rag/search - Pipeline complet", resp)
data = resp.json()
assert "answer" in data and "strategy_info" in data and "duration_ms" in data
print(f"\n  answer: {data['answer'][:200]}...")
print(f"  citations: {len(data['citations'])}")
print(f"  results: {len(data['results'])}")
print(f"  strategy: {data['strategy_info']}")


  POST /api/v1/rag/search - Pipeline complet
  Status: 200
  X-Process-Time: 67.8256s
  Body:
{
  "answer": "",
  "query": "Quelles sont les causes fréquentes de panne sur les pompes ?",
  "citations": [],
  "results": [
    {
      "chunk_id": "119",
      "content": "cause: Présence d'eau condensée dans le circuit d'air et grippage du tiroir.\nsolution: Purge du réseau d'air comprimé, démontage et lubrification du tiroir de la vanne.\nsymptomes: Absence d'actionneur sur la ligne, défaut de pression pneumatique.",
      "source_name": "panne:9",
      "source_type": "panne",
      "retrieval_score": 0.8438145,
      "rerank_score": 0.02481813356280327,
      "rank": 1,
      "id_document": null,
      "id_panne": 9,
      "id_equipement": 405,
      "retrieval_strategy": "qdrant",
      "reranker_strategy": "cross-encoder"
    },
    {
      "chunk_id": "98",
      "content": "cause: Usure prématurée du joint d'étanchéité principal.\nsolution: Remplacement du kit de joints d'étanchéi

In [39]:
# --- Cas 7.2 : Sans reranking ---
# Attendu: 200 + results avec reranker_strategy="none"
payload = {"query": "vibration moteur", "rerank": False, "generate": True}
resp = requests.post(f"{BASE_URL}/api/v1/rag/search", headers=HEADERS, json=payload)
show("POST /api/v1/rag/search - Sans rerank", resp)


  POST /api/v1/rag/search - Sans rerank
  Status: 200
  X-Process-Time: 2.2320s
  Body:
{
  "answer": "",
  "query": "vibration moteur",
  "citations": [],
  "results": [
    {
      "chunk_id": "112",
      "content": "--- Row 1 ---\nid_panne: 7\ntitre: Vibration anormale sur pompe centrifuge\ndescription: Niveau sonore excessif et secousses importantes mesurées sur le corps de pompe.\ngravite: Moyenne\ndate_detection: 2026-06-07 15:50:00",
      "source_name": "panne:7",
      "source_type": "panne",
      "retrieval_score": 0.85995054,
      "rerank_score": 0.85995054,
      "rank": 1,
      "id_document": null,
      "id_panne": 7,
      "id_equipement": 224,
      "retrieval_strategy": "qdrant",
      "reranker_strategy": "none"
    },
    {
      "chunk_id": "105",
      "content": "moteur en surcharge thermique.\nstatut_indexation: Indexe\nid_equipement: 412\nid_ot: 5045",
      "source_name": "panne:4",
      "source_type": "panne",
      "retrieval_score": 0.8405071,
      "r

In [40]:
# --- Cas 7.3 : Sans génération LLM ---
# Attendu: 200 + answer="", citations=[]
payload = {"query": "maintenance préventive", "rerank": True, "generate": False}
resp = requests.post(f"{BASE_URL}/api/v1/rag/search", headers=HEADERS, json=payload)
show("POST /api/v1/rag/search - Sans generate", resp)
data = resp.json()
assert data["answer"] == ""


  POST /api/v1/rag/search - Sans generate
  Status: 200
  X-Process-Time: 5.1512s
  Body:
{
  "answer": "",
  "query": "maintenance préventive",
  "citations": [],
  "results": [
    {
      "chunk_id": "95",
      "content": "detection: 2026-06-01 08:30:00\ncause: Encrassement des échangeurs thermiques et défaillance du ventilateur de refroidissement.\nsolution: Nettoyage complet des échangeurs et remplacement du ventilateur défectueux.",
      "source_name": "panne:1",
      "source_type": "panne",
      "retrieval_score": 0.8208796,
      "rerank_score": 0.0013292984804138541,
      "rank": 1,
      "id_document": null,
      "id_panne": 1,
      "id_equipement": 102,
      "retrieval_strategy": "qdrant",
      "reranker_strategy": "cross-encoder"
    },
    {
      "chunk_id": "89",
      "content": "detection: 2026-06-01 08:30:00\ncause: Encrassement des échangeurs thermiques et défaillance du ventilateur de refroidissement.\nsolution: Nettoyage complet des échangeurs et remplace

In [41]:
# --- Cas 7.4 : Avec filtres ---
payload = {
    "query": "défaut",
    "filters": {"source_type": "panne"},
    "top_k": 3, "rerank": True, "generate": True
}
resp = requests.post(f"{BASE_URL}/api/v1/rag/search", headers=HEADERS, json=payload)
show("POST /api/v1/rag/search - Avec filtres", resp)


  POST /api/v1/rag/search - Avec filtres
  Status: 200
  X-Process-Time: 6.1524s
  Body:
{
  "answer": "",
  "query": "défaut",
  "citations": [],
  "results": [
    {
      "chunk_id": "90",
      "content": "ent du ventilateur défectueux.\nsymptomes: Montée rapide de la température au-delà de 90°C, arrêt automatique de sécurité.\nstatut_indexation: Indexe\nid_equipement: 102\nid_ot: 5041",
      "source_name": "panne:1",
      "source_type": "panne",
      "retrieval_score": 0.83566964,
      "rerank_score": 0.0332159660756588,
      "rank": 1,
      "id_document": null,
      "id_panne": 1,
      "id_equipement": 102,
      "retrieval_strategy": "qdrant",
      "reranker_strategy": "cross-encoder"
    },
    {
      "chunk_id": "96",
      "content": "ent du ventilateur défectueux.\nsymptomes: Montée rapide de la température au-delà de 90°C, arrêt automatique de sécurité.\nstatut_indexation: Indexe\nid_equipement: 102\nid_ot: 5041",
      "source_name": "panne:1",
      "source_typ

In [42]:
# --- Cas 7.5 : Query vide ---
# Attendu: 422
resp = requests.post(f"{BASE_URL}/api/v1/rag/search", headers=HEADERS, json={"query": ""})
show("POST /api/v1/rag/search - Query vide", resp)
assert resp.status_code == 422

# --- Cas 7.6 : Sans auth ---
resp = requests.post(f"{BASE_URL}/api/v1/rag/search", json={"query": "test"})
show("POST /api/v1/rag/search - Sans auth", resp)

# --- Cas 7.7 : top_k hors limites ---
# Attendu: 422
resp = requests.post(f"{BASE_URL}/api/v1/rag/search", headers=HEADERS,
    json={"query": "test", "top_k": -1})
show("POST /api/v1/rag/search - top_k=-1", resp)
assert resp.status_code == 422


  POST /api/v1/rag/search - Query vide
  Status: 422
  X-Process-Time: 0.0060s
  Body:
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "body",
        "query"
      ],
      "msg": "String should have at least 1 character",
      "input": "",
      "ctx": {
        "min_length": 1
      }
    }
  ]
}

  POST /api/v1/rag/search - Sans auth
  Status: 422
  X-Process-Time: 0.0245s
  Body:
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "header",
        "Authorization"
      ],
      "msg": "Field required",
      "input": null
    }
  ]
}

  POST /api/v1/rag/search - top_k=-1
  Status: 422
  X-Process-Time: 0.0041s
  Body:
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "body",
        "top_k"
      ],
      "msg": "Input should be greater than or equal to 1",
      "input": -1,
      "ctx": {
        "ge": 1
      }
    }
  ]
}


---
## 8. Ingestion - Fichier (POST /api/v1/ingest/file)

**Auth requise.** Upload multipart/form-data.
Le fichier est envoyé comme champ `file` du body (form-data), avec des métadonnées optionnelles.

In [46]:
import os 
print("the current w . d : ",os.getcwd())

the current w . d :  /home/abdellah-daif/GMAO-RAG/tests/manual


In [54]:
# --- Cas 8.1 : Créer un fichier test ---
test_file = Path("/tmp/test_manual_api.txt")
test_file.write_text(
    "Ceci est un test d'ingestion via l'API REST.\n"
    "La pompe centrifuge N°5 présente des vibrations excessives.\n"
    "L'intervention a été effectuée le 15/03/2024.\n"
    "Cause: déséquilibre du rotor.\n"
    "Solution: rééquilibrage et remplacement des paliers.",
    encoding="utf-8"
)
print(f"Fichier test créé: {test_file}")

Fichier test créé: /tmp/test_manual_api.txt


In [55]:
# --- Cas 8.2 : Upload avec métadonnées ---
# Attendu: 200 + IngestResponse
auth_headers = {"Authorization": f"Bearer {API_KEY}"}

with open(test_file, "rb") as f:
    files = {"file": ("test_manual_api.txt", f, "text/plain")}
    data = {"id_equipement": "42", "chunk_size": "300", "chunk_overlap": "30"}
    resp = requests.post(f"{BASE_URL}/api/v1/ingest/file", headers=auth_headers, files=files, data=data)

show("POST /api/v1/ingest/file - Upload avec métadonnées", resp)
ingest_data = resp.json()
print(f"  -> status={ingest_data['status']}, chunks={ingest_data['results'][0]['chunks_count']}")


  POST /api/v1/ingest/file - Upload avec métadonnées
  Status: 200
  X-Process-Time: 0.6477s
  Body:
{
  "status": "ok",
  "results": [
    {
      "status": "ok",
      "document_name": "tmpi0jcotsg.txt",
      "chunks_count": 1,
      "duration_ms": 636.01,
      "error": null
    }
  ],
  "total_files": 1,
  "success_count": 1,
  "error_count": 0
}
  -> status=ok, chunks=1


In [49]:
# --- Cas 8.3 : Upload sans métadonnées (tout par défaut) ---
with open(test_file, "rb") as f:
    files = {"file": ("test_manual_api.txt", f, "text/plain")}
    resp = requests.post(f"{BASE_URL}/api/v1/ingest/file", headers=auth_headers, files=files)

show("POST /api/v1/ingest/file - Sans métadonnées", resp)


  POST /api/v1/ingest/file - Sans métadonnées
  Status: 200
  X-Process-Time: 0.3001s
  Body:
{
  "status": "partial",
  "results": [
    {
      "status": "error",
      "document_name": "/tmp/tmplsdjtthi.txt",
      "chunks_count": 0,
      "duration_ms": 290.32,
      "error": "[PARTIAL_STORAGE_ERROR] Storage completed only partially."
    }
  ],
  "total_files": 1,
  "success_count": 0,
  "error_count": 1
}


In [ ]:
# --- Cas 8.4 : Upload sans auth ---
# Attendu: 401 ou 422
with open(test_file, "rb") as f:
    files = {"file": ("test_manual_api.txt", f, "text/plain")}
    resp = requests.post(f"{BASE_URL}/api/v1/ingest/file", files=files)

show("POST /api/v1/ingest/file - Sans auth", resp)

In [ ]:
# --- Cas 8.5 : Upload fichier vide ---
# Le pipeline va échouer (aucun chunkable content)
files = {"file": ("empty.txt", b"", "text/plain")}
resp = requests.post(f"{BASE_URL}/api/v1/ingest/file", headers=auth_headers, files=files)
show("POST /api/v1/ingest/file - Fichier vide", resp)

---
## 9. Ingestion - MySQL (POST /api/v1/ingest/database)

**Auth requise.** Ingestion depuis une table MySQL ou une requête SQL custom.
Les identifiants de connexion sont fournis dans le body JSON.

In [51]:
# --- Cas 9.1 : Ingestion depuis une table ---
# Attendu: 200 + IngestResponse
payload = {
    "driver": "mysql",
    "host": "127.0.0.1",
    "port": 3306,
    "database": "gmao_rag",
    "user": "root",
    "password": "Zzdv6401",
    "table": "panne",
    "id_equipement": 1,
    "chunk_size": 500,
    "chunk_overlap": 50
}
resp = requests.post(f"{BASE_URL}/api/v1/ingest/database", headers=HEADERS, json=payload)
show("POST /api/v1/ingest/database - Table simple", resp)


  POST /api/v1/ingest/database - Table simple
  Status: 200
  X-Process-Time: 2.8774s
  Body:
{
  "status": "partial",
  "results": [
    {
      "status": "error",
      "document_name": "{'driver': 'mysql', 'host': '127.0.0.1', 'port': 3306, 'database': 'gmao_rag', 'user': 'root', 'password': 'Zzdv6401', 'table': 'panne'}",
      "chunks_count": 0,
      "duration_ms": 2865.27,
      "error": "[PARTIAL_STORAGE_ERROR] Storage completed only partially."
    }
  ],
  "total_files": 1,
  "success_count": 0,
  "error_count": 1
}


In [ ]:
# --- Cas 9.2 : Requête SQL custom ---
payload = {
    "driver": "mysql",
    "host": "127.0.0.1",
    "port": 3306,
    "database": "gmao_rag",
    "user": "root",
    "password": "",
    "table": "panne",
    "query": "SELECT * FROM panne LIMIT 5",
    "id_equipement": 1
}
resp = requests.post(f"{BASE_URL}/api/v1/ingest/database", headers=HEADERS, json=payload)
show("POST /api/v1/ingest/database - Requête SQL custom", resp)

In [ ]:
# --- Cas 9.3 : Host vide ---
# Attendu: 422 (min_length=1)
payload = {"host": "", "database": "test", "user": "root", "password": "", "table": "test"}
resp = requests.post(f"{BASE_URL}/api/v1/ingest/database", headers=HEADERS, json=payload)
show("POST /api/v1/ingest/database - Host vide", resp)
assert resp.status_code == 422

# --- Cas 9.4 : Port invalide ---
# Attendu: 422 (gt=0)
payload = {"host": "localhost", "port": -1, "database": "test", "user": "root", "password": "", "table": "test"}
resp = requests.post(f"{BASE_URL}/api/v1/ingest/database", headers=HEADERS, json=payload)
show("POST /api/v1/ingest/database - Port négatif", resp)
assert resp.status_code == 422

# --- Cas 9.5 : Sans auth ---
payload = {"host": "localhost", "database": "test", "user": "root", "password": "", "table": "test"}
resp = requests.post(f"{BASE_URL}/api/v1/ingest/database", json=payload)
show("POST /api/v1/ingest/database - Sans auth", resp)

---
## 10. Ingestion - Batch (POST /api/v1/ingest/files)

**Auth requise.** Ingestion de plusieurs fichiers déjà présents sur le serveur (par chemin absolu).

In [ ]:
# --- Cas 10.1 : Batch - un seul fichier ---
# Attendu: 200 + IngestResponse (total_files=1)
payload = {
    "paths": [str(test_file)],
    "id_equipement": 42,
    "chunk_size": 300,
    "chunk_overlap": 30
}
resp = requests.post(f"{BASE_URL}/api/v1/ingest/files", headers=HEADERS, json=payload)
show("POST /api/v1/ingest/files - Un fichier", resp)
data = resp.json()
assert data["total_files"] == 1

In [ ]:
# --- Cas 10.2 : Batch - deux fichiers ---
test_file2 = Path("/tmp/test_manual_api_2.txt")
test_file2.write_text(
    "Fichier de test numéro deux.\n"
    "Le moteur électrique tourne à 1500 tr/min.\n"
    "Température ambiante: 25°C.",
    encoding="utf-8"
)
payload = {"paths": [str(test_file), str(test_file2)], "id_equipement": 42}
resp = requests.post(f"{BASE_URL}/api/v1/ingest/files", headers=HEADERS, json=payload)
show("POST /api/v1/ingest/files - Deux fichiers", resp)
data = resp.json()
print(f"  -> total={data['total_files']}, ok={data['success_count']}, errors={data['error_count']}")

In [ ]:
# --- Cas 10.3 : Paths vides ---
# Attendu: 422 (min_length=1)
resp = requests.post(f"{BASE_URL}/api/v1/ingest/files", headers=HEADERS, json={"paths": []})
show("POST /api/v1/ingest/files - Paths vides", resp)
assert resp.status_code == 422

# --- Cas 10.4 : Chemin inexistant ---
# Attendu: 200 + status="error" (le fichier n'existe pas)
payload = {"paths": ["/chemin/qui/nexiste/pas.txt"]}
resp = requests.post(f"{BASE_URL}/api/v1/ingest/files", headers=HEADERS, json=payload)
show("POST /api/v1/ingest/files - Chemin inexistant", resp)

---
## 11. Résumé des tests

### Scénarios testés par endpoint

| Endpoint | Cas succès | Cas erreur |
|----------|------------|------------|
| GET /health | Health check | — |
| GET /strategies | Auth valide | Sans auth, mauvais token, header mal formé |
| GET /stats | Stats normales | Sans auth |
| GET /documents | Liste, détail existant | Inexistant, ID invalide |
| DELETE /documents | — | Inexistant |
| POST /retrieve | Simple, filtres | Query vide, top_k 0/100, sans auth, body manquant |
| POST /rerank | Normal | Candidats vides, query vide, champs manquants |
| POST /search | Complet, sans rerank, sans generate, filtres | Query vide, sans auth, top_k invalide |
| POST /ingest/file | Upload avec/sans métadonnées | Sans auth, fichier vide |
| POST /ingest/database | Table, SQL custom | Host vide, port invalide, sans auth |
| POST /ingest/files | 1 fichier, 2 fichiers | Paths vides, chemin inexistant |

### Codes de réponse attendus

| Code | Signification |
|------|---------------|
| 200 | Succès |
| 401 | Token invalide ou header mal formé |
| 422 | Validation error (champ requis, valeur hors limites) |
| 404 | Document non trouvé |
| 500 | Erreur serveur interne |

In [ ]:
# --- Nettoyage ---
for f in [test_file, test_file2]:
    if f.exists():
        f.unlink()
        print(f"Supprimé: {f}")
print("\nNettoyage terminé.")